# 🎩 The Butler's Masterclass: Phase 3 Machine Learning Decision Engine

*From: The Head of the Project*
*To: The Maestro (User)*

## Briefing
Phase 2 is strictly controlled. The Polarity Bug is dead. The erasures and printed letters are filtered aggressively by the 0.60x inner mask and the penalizing solidity check. We have yielded a phenomenally clean dataset of `FILLED`, `BLANK`, and true `UNCERTAIN` edge cases.

The remaining lift is to funnel this immaculate dataset into a Deep Learning model that acts as our ultimate arbiter (The Decision Engine) for Row Scoring. We need models that hit OMR standards (Scantron/ZipGrade level). 

As instructed, I have prepared **Three Ascendant Architectures** in PyTorch for your immediate execution:

1.  **The Diamond/Pyramid Architecture:** An aggressive feature extractor that expands channel depth then strictly contracts.
2.  **The Ascending Architecture:** A relentless, widening Convolutional Neural Network (CNN) funneling into a massive dense decision layer.
3.  **Transfer Learning (The Heavy Artillery):** A pre-trained ResNet-18 model, modified to annihilate this 64x64 classification task.

Execute the cells below sequentially. The Butler has set the table. You simply need to dine.

## Run Order (Overhaul-Safe)

1. Confirm dependencies in your ML environment:
   - `pip install torch torchvision numpy pyyaml tqdm`
2. Prepare dataset in either mode:
   - split mode: `<dataset_root>/train/<class>` and `<dataset_root>/valid/<class>`
   - single-folder mode: `<dataset_root>/<class>` (not ideal, notebook will auto-split)
3. Run cells from top to bottom.
4. Review outputs under `Phase2/papertrail/notebook/`:
   - `training_epoch_metrics.jsonl`
   - `training_summary.json`
   - `butler_best_model.pth`
5. If data ingestion fails, fix paths in Cell 2 and rerun Cell 2 onward.

## Command Center: Variables & Paths

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()

# List of possible dataset locations
CANDIDATE_ROOTS = [
    PROJECT_ROOT / 'ModelBackEnd' / 'SwiftGrade_Datasets',
    PROJECT_ROOT / 'SwiftGrade_Datasets',
    PROJECT_ROOT / 'SwiftGradeOMRv2 - Trial3' / 'Phase2' / 'manual_labeled',
]

In [ ]:
# ==========================================
# MANUAL DATASET OVERRIDE (Optional)
# ==========================================
# If you need to manually specify a different path, uncomment below:
# DATASET_ROOT = Path(r"C:\Path\To\Your\Dataset")

In [ ]:
# ==========================================
# DATASET RESOLVER & MODE DETECTION
# ==========================================
if 'DATASET_ROOT' not in locals() or DATASET_ROOT is None:
    DATASET_ROOT = None
    for candidate in CANDIDATE_ROOTS:
        if candidate.exists():
            DATASET_ROOT = candidate
            break
    if DATASET_ROOT is None:
        DATASET_ROOT = CANDIDATE_ROOTS[0]

TRAIN_DIR = DATASET_ROOT / 'train'
VALID_DIR = DATASET_ROOT / 'valid'

CLASS_FOLDER_NAMES = {'filled', 'blank', 'uncertain'}
root_class_dirs = []
if DATASET_ROOT.exists():
    root_class_dirs = [
        p.name for p in DATASET_ROOT.iterdir()
        if p.is_dir() and p.name.lower() in CLASS_FOLDER_NAMES
    ]

if TRAIN_DIR.exists() and VALID_DIR.exists():
    DATASET_MODE = 'split'
elif len(root_class_dirs) >= 2:
    DATASET_MODE = 'single_folder'
else:
    DATASET_MODE = 'missing'

print(f"[Config] DATASET_ROOT: {DATASET_ROOT}")
print(f"[Config] DATASET_MODE: {DATASET_MODE}")

In [ ]:
# ==========================================
# HYPERPARAMETERS & LOGGING
# ==========================================
BATCH_SIZE = 32
NUM_EPOCHS = 15
LEARNING_RATE = 0.001
NUM_WORKERS = 0  # Windows-safe default
VAL_SPLIT_RATIO = 0.2
RANDOM_SPLIT_SEED = 42

POSITIVE_CLASS_NAME = 'FILLED'
UNCERTAIN_CLASS_NAME = 'UNCERTAIN'

PAPERTRAIL_DIR = PROJECT_ROOT / 'SwiftGradeOMRv2 - Trial3' / 'Phase2' / 'papertrail' / 'notebook'
PAPERTRAIL_DIR.mkdir(parents=True, exist_ok=True)

print(f"[Config] PAPERTRAIL_DIR: {PAPERTRAIL_DIR}")

In [ ]:
# Cell 1: Environment & Core Imports
import json
from datetime import datetime, timezone
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import numpy as np

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[Runtime] Device set to: {device}")

## Step 1: Data Ingestion (From Phase 2's Strict Output)

In [ ]:
# Cell 2: Data Loading with class-balance diagnostics

train_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def compute_class_weights(dataset_obj, num_classes):
    class_counts = defaultdict(int)

    if hasattr(dataset_obj, 'samples'):
        for _path, class_idx in dataset_obj.samples:
            class_counts[int(class_idx)] += 1
    elif hasattr(dataset_obj, 'targets'):
        for class_idx in dataset_obj.targets:
            class_counts[int(class_idx)] += 1
    elif hasattr(dataset_obj, 'indices') and hasattr(dataset_obj, 'dataset') and hasattr(dataset_obj.dataset, 'targets'):
        for idx in dataset_obj.indices:
            class_counts[int(dataset_obj.dataset.targets[idx])] += 1
    else:
        raise RuntimeError('Unsupported dataset type for class-weight computation.')

    total = sum(class_counts.values())
    weights = []
    for class_idx in range(num_classes):
        count = class_counts.get(class_idx, 1)
        weights.append(total / float(num_classes * max(count, 1)))

    return class_counts, torch.tensor(weights, dtype=torch.float32)

def build_train_val_datasets():
    if DATASET_MODE == 'split':
        train_ds = datasets.ImageFolder(root=str(TRAIN_DIR), transform=train_transforms)
        val_ds = datasets.ImageFolder(root=str(VALID_DIR), transform=val_transforms)

        if train_ds.class_to_idx != val_ds.class_to_idx:
            raise RuntimeError('train/valid class_to_idx mismatch. Ensure both splits contain same class folders.')

        return train_ds, val_ds, dict(train_ds.class_to_idx), 'split'

    if DATASET_MODE == 'single_folder':
        base_train = datasets.ImageFolder(root=str(DATASET_ROOT), transform=train_transforms)
        base_val = datasets.ImageFolder(root=str(DATASET_ROOT), transform=val_transforms)
        n_items = len(base_train)

        if n_items < 2:
            raise RuntimeError('Need at least 2 samples in single_folder mode to split train/valid.')

        val_size = max(1, int(round(n_items * VAL_SPLIT_RATIO)))
        train_size = n_items - val_size
        if train_size <= 0:
            train_size = n_items - 1
            val_size = 1

        generator = torch.Generator().manual_seed(RANDOM_SPLIT_SEED)
        perm = torch.randperm(n_items, generator=generator).tolist()
        val_idx = perm[:val_size]
        train_idx = perm[val_size:]

        train_ds = Subset(base_train, train_idx)
        val_ds = Subset(base_val, val_idx)

        return train_ds, val_ds, dict(base_train.class_to_idx), 'single_folder_random_split'

    raise RuntimeError(
        f'Dataset mode is missing. Checked root={DATASET_ROOT}. Expected either train/valid folders or class folders.'
    )

train_dataset = None
val_dataset = None
train_loader = None
val_loader = None
class_counts = {}
class_weights = None
class_to_idx = {}
idx_to_class = {}
class_names = []

try:
    train_dataset, val_dataset, class_to_idx, resolved_mode = build_train_val_datasets()

    idx_to_class = {v: k for k, v in class_to_idx.items()}
    class_names = [name for name, _idx in sorted(class_to_idx.items(), key=lambda kv: kv[1])]

    class_counts, class_weights = compute_class_weights(train_dataset, num_classes=len(class_names))

    print(f"[Data] Mode:               {resolved_mode}")
    print(f"[Data] Training Samples:   {len(train_dataset)}")
    print(f"[Data] Validation Samples: {len(val_dataset)}")
    print(f"[Data] Classes:            {class_names}")
    print(f"[Data] Class Counts:       {dict(class_counts)}")
    print(f"[Data] Class Weights:      {class_weights.tolist()}")

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    print("[Data] DataLoaders initialized.")
except Exception as e:
    print(f"[ERROR] Data ingestion failed. Ensure dataset paths and class folders are valid. Error: {e}")

## Step 2: The Architectures (Choose Your Weapon)

In [ ]:
# Architecture 1: The Diamond / Pyramid (7 Layers)
# Strategy: Expand to peak channels then contract. Deep feature extraction with forced dimensionality reduction.
class DiamondCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(DiamondCNN, self).__init__()
        # Expansion phase (3 layers) — steadily widen channels
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2)
        ) # Output: 32 x 32 x 32
        
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        ) # Output: 64 x 16 x 16
        
        self.layer3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        ) # Output: 128 x 8 x 8
        
        # Peak (1 layer at peak capacity)
        self.layer4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU()
        ) # Output: 256 x 8 x 8 (no pooling at peak)
        
        # Contraction phase (3 layers) — narrow channels back down
        self.layer5 = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        ) # Output: 128 x 4 x 4
        
        self.layer6 = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        ) # Output: 64 x 4 x 4
        
        self.layer7 = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        ) # Output: 32 x 4 x 4
        
        self.fc = nn.Sequential(
            nn.Linear(32 * 4 * 4, 256),
            nn.Dropout(0.5),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.layer5(out)
        out = self.layer6(out)
        out = self.layer7(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

In [ ]:
# Architecture 2: The Ascending Network (7 Layers)
# Strategy: Continuously widen across 7 conv layers. Build capacity steadily with BatchNorm stabilization.
class AscendingCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(AscendingCNN, self).__init__()
        self.features = nn.Sequential(
            # Layer 1: 16 channels
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 32x32
            
            # Layer 2: 32 channels
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 16x16
            
            # Layer 3: 64 channels
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 8x8
            
            # Layer 4: 128 channels
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 4x4
            
            # Layer 5: 256 channels
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            
            # Layer 6: 512 channels
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            
            # Layer 7: 1024 channels (peak width)
            nn.Conv2d(512, 1024, kernel_size=3, padding=1),
            nn.BatchNorm2d(1024),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))  # Collapse to 1x1
        )
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.Dropout(0.5),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.Dropout(0.3),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

In [ ]:
# Architecture 3: Transfer Learning (Heavy Artillery - ResNet18)
# Strategy: Stand on the shoulders of giants. Fine-tune a model trained on millions of images.
def get_transfer_learning_model(num_classes=3):
    # Keep compatibility across torchvision versions.
    try:
        weights = models.ResNet18_Weights.DEFAULT
        model = models.resnet18(weights=weights)
    except Exception:
        # Fallback for older torchvision or offline weight loading constraints.
        try:
            model = models.resnet18(pretrained=True)
        except Exception:
            model = models.resnet18(weights=None)

    # Freeze the early layers to retain generic feature extractors (edges, blobs)
    for param in model.parameters():
        param.requires_grad = False

    # Replace final layer for our classes.
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    return model

## Step 3: The Execution Engine

In [ ]:
# Precision-first training loop with papertrail metrics

def resolve_class_index(name, default_idx=0):
    for class_name, class_idx in class_to_idx.items():
        if class_name.lower() == name.lower():
            return class_idx
    return default_idx

def compute_metrics(labels, preds, num_classes, positive_idx, uncertain_idx=None):
    confusion = [[0 for _ in range(num_classes)] for _ in range(num_classes)]
    for truth, pred in zip(labels, preds):
        confusion[truth][pred] += 1

    total = len(labels)
    correct = sum(confusion[i][i] for i in range(num_classes))
    accuracy = correct / total if total else 0.0

    tp = sum(1 for t, p in zip(labels, preds) if t == positive_idx and p == positive_idx)
    fp = sum(1 for t, p in zip(labels, preds) if t != positive_idx and p == positive_idx)
    fn = sum(1 for t, p in zip(labels, preds) if t == positive_idx and p != positive_idx)
    tn = sum(1 for t, p in zip(labels, preds) if t != positive_idx and p != positive_idx)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0

    uncertain_rate = 0.0
    if uncertain_idx is not None:
        uncertain_rate = sum(1 for p in preds if p == uncertain_idx) / total if total else 0.0

    return {
        'accuracy': accuracy,
        'precision_filled': precision,
        'recall_filled': recall,
        'f1_filled': f1,
        'fpr_filled': fpr,
        'uncertain_rate': uncertain_rate,
        'confusion': confusion,
    }

def evaluate_model(model, loader, criterion, num_classes, positive_idx, uncertain_idx=None):
    model.eval()
    running_loss = 0.0
    labels_all = []
    preds_all = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            running_loss += loss.item() * inputs.size(0)
            labels_all.extend(labels.detach().cpu().numpy().tolist())
            preds_all.extend(preds.detach().cpu().numpy().tolist())

    mean_loss = running_loss / len(loader.dataset) if len(loader.dataset) else 0.0
    metrics = compute_metrics(labels_all, preds_all, num_classes, positive_idx, uncertain_idx)
    metrics['loss'] = mean_loss
    return metrics

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    if train_loader is None or val_loader is None:
        raise RuntimeError('DataLoaders are not initialized. Check data ingestion cell output.')

    model = model.to(device)
    num_classes = len(class_to_idx) if class_to_idx else 2
    positive_idx = resolve_class_index(POSITIVE_CLASS_NAME, default_idx=0)
    uncertain_idx = resolve_class_index(UNCERTAIN_CLASS_NAME, default_idx=None) if class_to_idx else None

    best_key = -1.0
    history = []
    best_model_path = PAPERTRAIL_DIR / 'butler_best_model.pth'
    epoch_metrics_path = PAPERTRAIL_DIR / 'training_epoch_metrics.jsonl'

    with open(epoch_metrics_path, 'w', encoding='utf-8') as metrics_file:
        for epoch in range(num_epochs):
            print(f'\nEpoch {epoch + 1}/{num_epochs}')
            print('-' * 10)

            model.train()
            running_loss = 0.0
            train_labels = []
            train_preds = []

            for inputs, labels in train_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)

                loss.backward()
                optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                train_labels.extend(labels.detach().cpu().numpy().tolist())
                train_preds.extend(preds.detach().cpu().numpy().tolist())

            train_metrics = compute_metrics(train_labels, train_preds, num_classes, positive_idx, uncertain_idx)
            train_metrics['loss'] = running_loss / len(train_loader.dataset) if len(train_loader.dataset) else 0.0

            val_metrics = evaluate_model(
                model=model,
                loader=val_loader,
                criterion=criterion,
                num_classes=num_classes,
                positive_idx=positive_idx,
                uncertain_idx=uncertain_idx,
            )

            epoch_row = {
                'epoch': epoch + 1,
                'train_loss': train_metrics['loss'],
                'train_accuracy': train_metrics['accuracy'],
                'train_precision_filled': train_metrics['precision_filled'],
                'train_recall_filled': train_metrics['recall_filled'],
                'train_f1_filled': train_metrics['f1_filled'],
                'train_fpr_filled': train_metrics['fpr_filled'],
                'val_loss': val_metrics['loss'],
                'val_accuracy': val_metrics['accuracy'],
                'val_precision_filled': val_metrics['precision_filled'],
                'val_recall_filled': val_metrics['recall_filled'],
                'val_f1_filled': val_metrics['f1_filled'],
                'val_fpr_filled': val_metrics['fpr_filled'],
                'val_uncertain_rate': val_metrics['uncertain_rate'],
                'val_confusion': val_metrics['confusion'],
            }
            history.append(epoch_row)
            metrics_file.write(json.dumps(epoch_row) + '\n')

            print(
                f"Train Loss: {train_metrics['loss']:.4f} | Val Loss: {val_metrics['loss']:.4f} | "
                f"Val Precision(FILLED): {val_metrics['precision_filled']:.4f} | "
                f"Val Recall(FILLED): {val_metrics['recall_filled']:.4f} | "
                f"Val FPR(FILLED): {val_metrics['fpr_filled']:.4f}"
            )

            # Precision-first model selection.
            selection_key = (val_metrics['precision_filled'], val_metrics['f1_filled'])
            numeric_key = selection_key[0] * 1000.0 + selection_key[1]
            if numeric_key > best_key:
                best_key = numeric_key
                torch.save(model.state_dict(), best_model_path)

    summary_path = PAPERTRAIL_DIR / 'training_summary.json'
    summary_payload = {
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
        'epochs': num_epochs,
        'best_model_path': str(best_model_path),
        'metrics_log_path': str(epoch_metrics_path),
        'history': history,
    }
    with open(summary_path, 'w', encoding='utf-8') as summary_file:
        json.dump(summary_payload, summary_file, indent=2)

    print(f"\n[Training] complete. Best model saved to: {best_model_path}")
    print(f"[Training] metrics log: {epoch_metrics_path}")
    print(f"[Training] summary: {summary_path}")
    return model, history

## Launch Sequence
Uncomment the architecture you want and execute the final code cell.

This notebook writes a papertrail under `Phase2/papertrail/notebook/`:
- `training_epoch_metrics.jsonl` (epoch-level metrics)
- `training_summary.json`
- `butler_best_model.pth`

Recommended execution workflow:
1. Run Cell 1 (intro) and Cell 2 (run-order notes).
2. Run Cell 3 (config) and confirm `DATASET_MODE` is `split` or `single_folder`.
3. Run Cell 4 (imports), then Cell 5 (data loading).
4. Run architecture cells (Cells 7, 8, 9) to define models.
5. Run Cell 11 (training engine), then Cell 13 (launch).
6. Review papertrail metrics; optimize for `Val Precision(FILLED)` and `Val FPR(FILLED)` before promoting model.

In [ ]:
# Launch Sequence

NUM_CLASSES = len(class_to_idx) if class_to_idx else 3

model = DiamondCNN(num_classes=NUM_CLASSES)
# model = AscendingCNN(num_classes=NUM_CLASSES)
# model = get_transfer_learning_model(num_classes=NUM_CLASSES)

if class_weights is not None:
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    print('[Launch] Using class-weighted CrossEntropyLoss.')
else:
    criterion = nn.CrossEntropyLoss()
    print('[Launch] Using unweighted CrossEntropyLoss.')

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print('[Launch] Engaging training drive...')
try:
    trained_model, training_history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        num_epochs=NUM_EPOCHS,
    )
    print(f"[Launch] Completed. Epochs logged: {len(training_history)}")
except Exception as e:
    print(f"[ERROR] Training loop aborted: {e}")